# 18 Union

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Stack DataFrames vertically with <code>union</code> and <code>unionByName</code>, handle schema differences (column order, missing columns, types), and know the related set operations: <code>intersect</code>, <code>exceptAll</code>, <code>subtract</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Pipelines constantly combine data with the same shape: daily files, regional extracts, history plus new data, or several sources mapped to one model. The classic bug is <code>union</code> matching columns <b>by position</b>, which silently puts values in the wrong column when the order differs. Knowing <code>unionByName</code> avoids it.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How union works</b>
<ul><li><code>union(other)</code> appends rows, matching columns <b>by position</b>. Names are ignored</li><li><code>unionByName(other)</code> matches columns <b>by name</b>. With <code>allowMissingColumns=True</code>, missing columns are filled with null</li><li>Both keep duplicates, like SQL <code>UNION ALL</code>. Add <code>.distinct()</code> for SQL <code>UNION</code> behavior</li><li>Both DataFrames need the same number of columns (except <code>unionByName</code> with missing columns allowed), and compatible types</li><li>Union is <b>narrow</b>: the partitions of both inputs are simply listed together, with no shuffle</li></ul>
</div>

```
df_a (3 partitions)  +  df_b (2 partitions)   ->   union: 5 partitions, no data movement
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A retailer receives sales extracts from three regional systems. The India system sends <code>store, date, amount</code>, the UAE system sends <code>date, store, amount, currency</code>, and the UK system sends <code>store, amount, date</code>. A positional <code>union</code> would put dates into the store column. <code>unionByName(allowMissingColumns=True)</code> lines everything up by name and fills the missing <code>currency</code> with null, to be set to a default afterwards.
</div>

## Syntax

```python
df1.union(df2)                                     # by position, keeps duplicates
df1.unionByName(df2)                               # by name
df1.unionByName(df2, allowMissingColumns=True)     # missing columns -> null
df1.union(df2).distinct()                          # SQL UNION (deduplicated)

from functools import reduce
reduce(lambda a, b: a.unionByName(b), list_of_dfs)  # many DataFrames

df1.intersect(df2), df1.exceptAll(df2), df1.subtract(df2)
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates sales extracts from three regions with different column orders, an extra column, and one overlapping row.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three small tables with different column orders. The India and UK data share the row <code>S1, 2024-08-01, 100.0</code>.
</div>

In [0]:
from pyspark.sql import functions as F

india = spark.createDataFrame(
    [("S1", "2024-08-01", 100.0), ("S2", "2024-08-01", 250.0)],
    "store STRING, sale_date STRING, amount DOUBLE",
)
uae = spark.createDataFrame(
    [("2024-08-01", "S7", 900.0, "AED"), ("2024-08-02", "S8", 75.0, "AED")],
    "sale_date STRING, store STRING, amount DOUBLE, currency STRING",
)
uk = spark.createDataFrame(
    [("S1", 100.0, "2024-08-01"), ("S5", 60.0, "2024-08-02")],
    "store STRING, amount DOUBLE, sale_date STRING",
)
india.show(); uae.show(); uk.show()

## 1. The positional union trap

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Unions India and UK data with <code>union</code>, where the column order differs.<br><br>
<b>Why it matters</b><br><code>union</code> doesn't look at names, only positions. The UK <code>amount</code> lands in <code>sale_date</code> and the date lands in <code>amount</code>. Under ANSI mode the implicit cast may fail, otherwise you get garbage silently. Either way, this is a bug.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Either an error about an invalid cast, or rows where <code>sale_date</code> holds <code>100.0</code> and <code>amount</code> holds a date. Both show the problem.
</div>

In [0]:
try:
    india.union(uk).show()
except Exception as e:
    print("union failed:", type(e).__name__, "-", str(e).splitlines()[0][:160])

## 2. unionByName

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Combines India and UK by column name.<br><br>
<b>Why it matters</b><br>This is the safe default whenever data comes from different sources. The output column order follows the <b>first</b> DataFrame.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 rows with every value in the right column. The duplicate <code>S1, 2024-08-01, 100.0</code> appears twice, because union keeps duplicates.
</div>

In [0]:
combined = india.unionByName(uk)
combined.show()
print("rows:", combined.count())

## 3. Missing columns with `allowMissingColumns`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds the UAE data, which has an extra <code>currency</code> column, and fills in a default.<br><br>
<b>Why it matters</b><br>Sources evolve: a new column appears in one extract before the others. <code>allowMissingColumns=True</code> keeps the pipeline running, and you decide how to fill the gaps.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
6 rows with a <code>currency</code> column: <code>AED</code> for UAE rows and <code>INR</code>/<code>GBP</code> defaults filled in for the others, based on the store.
</div>

In [0]:
all_regions = india.unionByName(uk).unionByName(uae, allowMissingColumns=True)
all_regions.show()

# Fill the defaults after the union. Here: S5 is the UK store, the rest are Indian.
all_regions = all_regions.withColumn(
    "currency",
    F.coalesce("currency", F.when(F.col("store") == "S5", "GBP").otherwise("INR")),
)
all_regions.orderBy("store").show()

## 4. Track where each row came from

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds a <code>source</code> column to each input before the union.<br><br>
<b>Why it matters</b><br>After a union you can't tell rows apart unless you label them. A <code>source</code> or <code>region</code> column is essential for debugging, auditing and reprocessing one source.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
6 rows, each labelled with its source (<code>india</code>, <code>uk</code> or <code>uae</code>), and a count per source.
</div>

In [0]:
labelled = (
    india.withColumn("source", F.lit("india"))
    .unionByName(uk.withColumn("source", F.lit("uk")))
    .unionByName(uae.withColumn("source", F.lit("uae")), allowMissingColumns=True)
)
labelled.groupBy("source").count().show()

## 5. Union many DataFrames, and deduplicate

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Combines a list of DataFrames with <code>functools.reduce</code>, then compares <code>UNION ALL</code> and <code>UNION</code> behavior.<br><br>
<b>Why it matters</b><br>Loops over files or partitions produce lists of DataFrames. <code>reduce</code> unions them in one expression. Remember that the DataFrame <code>union</code> is <code>UNION ALL</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
8 rows (4 DataFrames of 2 rows each), then 3 unique rows after <code>distinct()</code>.
</div>

In [0]:
from functools import reduce

frames = [india, uk.select("store", "sale_date", "amount"), india, uk.select("store", "sale_date", "amount")]
stacked = reduce(lambda a, b: a.unionByName(b), frames)
print("UNION ALL rows:", stacked.count())
print("UNION (distinct) rows:", stacked.distinct().count())

## 6. Type differences

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Unions a DataFrame where <code>amount</code> is a string with one where it is a double, then fixes it with an explicit cast.<br><br>
<b>Why it matters</b><br>Spark tries to find a common type. Depending on the types and ANSI settings, it may widen, cast, or fail. Cast explicitly before the union so the result type is a deliberate choice.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The implicit union succeeds here, because Spark finds a common type (<code>double</code>) and casts the string. A non-numeric string such as <code>'n/a'</code> would then fail at runtime under ANSI mode. The explicit cast gives the same schema, but makes the choice visible and testable.
</div>

In [0]:
as_text = spark.createDataFrame([("S9", "2024-08-03", "42.5")], "store STRING, sale_date STRING, amount STRING")

try:
    india.unionByName(as_text).printSchema()
except Exception as e:
    print("implicit union failed:", type(e).__name__)

fixed = india.unionByName(as_text.withColumn("amount", F.col("amount").cast("double")))
fixed.printSchema()
fixed.show()

## 7. Set operations: intersect, exceptAll, subtract

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Finds rows present in both DataFrames, and rows in one but not the other.<br><br>
<b>Why it matters</b><br>Useful for reconciliation and testing ("what changed between yesterday's and today's extract?"). <code>subtract</code> and <code>intersect</code> deduplicate (like SQL <code>EXCEPT DISTINCT</code> / <code>INTERSECT DISTINCT</code>), while <code>exceptAll</code> and <code>intersectAll</code> keep duplicates.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>intersect</code>: the shared <code>S1</code> row. <code>subtract</code> (India minus UK): the <code>S2</code> row. Then the SQL versions give the same results.
</div>

In [0]:
uk_aligned = uk.select("store", "sale_date", "amount")

india.intersect(uk_aligned).show()
india.subtract(uk_aligned).show()
india.exceptAll(uk_aligned).show()

india.createOrReplaceTempView("india_v")
uk_aligned.createOrReplaceTempView("uk_v")
spark.sql("SELECT * FROM india_v INTERSECT SELECT * FROM uk_v").show()
spark.sql("SELECT * FROM india_v EXCEPT SELECT * FROM uk_v").show()

## Under the hood

```
india (2 partitions) ─┐
                      ├─ Union ─> 2 + 2 + ... partitions, rows are NOT moved
uk    (2 partitions) ─┘

union().distinct()  -> Union + HashAggregate + Exchange   (the distinct shuffles)
intersect / subtract -> implemented as joins (left semi / left anti) + dedup -> shuffles
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> `union` / `unionByName` only concatenate partition lists, with no shuffle.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> `distinct`, `intersect` and `subtract` need a shuffle to compare rows.

Because union adds partitions, unioning many small DataFrames can create many tiny partitions and later many small files. Check the count with `spark_partition_id`, or let AQE coalesce them after a shuffle. Compare the plans below: no `Exchange` for the union, an `Exchange` for `intersect`.

In [0]:
a = spark.range(0, 1_000_000)
b = spark.range(500_000, 1_500_000)

print("---- union ----")
a.union(b).explain()

print("---- intersect ----")
a.intersect(b).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: values in the wrong columns after a union</b><br>
<code>union</code> matches by position. Use <code>unionByName</code>.<br><br>
**⛔ Problem: <code>NUM_COLUMNS_MISMATCH</code> / <code>[UNION...]</code> errors**<br>The DataFrames have a different number of columns. Use <code>unionByName(..., allowMissingColumns=True)</code> or select matching columns.<br><br>
<b>⛔ Problem: duplicate rows after a union</b><br>DataFrame <code>union</code> behaves like <code>UNION ALL</code>. Add <code>.distinct()</code> or deduplicate on a key.<br><br>
<b>⛔ Problem: incompatible types</b><br>Cast explicitly before the union, for example <code>amount</code> to <code>double</code> on both sides.<br><br>
<b>⛔ Problem: too many small partitions or files after unioning hundreds of DataFrames</b><br>Read the files together with one <code>spark.read</code> call (a folder or list of paths) instead of reading and unioning each one, or <code>coalesce</code> before writing.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>union</code> and <code>unionByName</code>?</b><br>
<code>union</code> matches columns by position and ignores names. <code>unionByName</code> matches by name, and with <code>allowMissingColumns=True</code> it fills missing columns with nulls. I use <code>unionByName</code> whenever the inputs come from different sources.<br><br>

<b>🎤 2. Does <code>union</code> remove duplicates?</b><br>
No. DataFrame <code>union</code> behaves like SQL <code>UNION ALL</code>. Use <code>.distinct()</code> to get SQL <code>UNION</code> semantics.<br><br>

<b>🎤 3. Does <code>union</code> cause a shuffle?</b><br>
No. It is a narrow transformation that just concatenates the partitions of both inputs. Only a following <code>distinct</code> would shuffle.<br><br>

<b>🎤 4. How do you combine a list of DataFrames?</b><br>
<code>functools.reduce(lambda a, b: a.unionByName(b), dfs)</code>. If they come from files, a single <code>spark.read</code> on all the paths is better.<br><br>

<b>🎤 5. What is the difference between <code>subtract</code> and <code>exceptAll</code>?</b><br>
<code>subtract</code> returns distinct rows from the first DataFrame that aren't in the second, like <code>EXCEPT DISTINCT</code>. <code>exceptAll</code> keeps duplicates, so it respects how many times each row appears.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Table <code>a</code> has 10 rows and table <code>b</code> has 10 rows, 3 of which are identical to rows in <code>a</code>. How many rows does <code>SELECT * FROM a UNION SELECT * FROM b</code> return, assuming no duplicates within each table?</b><br>
A. 20<br>
B. 17<br>
C. 10<br>
D. 3<br>
<details><summary><b>Show answer</b></summary><b>B.</b> SQL <code>UNION</code> removes duplicates: 10 + 10 − 3. <code>UNION ALL</code> would return 20.</details><br><br>

<b>Q2. Two DataFrames have the same columns in a different order. Which method combines them correctly?</b><br>
A. <code>df1.union(df2)</code><br>
B. <code>df1.unionAll(df2)</code><br>
C. <code>df1.unionByName(df2)</code><br>
D. <code>df1.join(df2, how="outer")</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> <code>union</code> and <code>unionAll</code> match by position.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Combine <code>india</code>, <code>uk</code> and <code>uae</code> into one DataFrame with the columns <code>store</code>, <code>sale_date</code> (as a date), <code>amount</code>, <code>currency</code> and <code>source</code></li><li>Default missing currencies to <code>INR</code> for India and <code>GBP</code> for UK, using the <code>source</code> column</li><li>Return the total amount per source</li><li>Find rows that appear in more than one source (same store, date and amount)</li><li>Using a set operation, list the India rows that are not in the UK data</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
from functools import reduce

# 1. Label each source, union by name, convert the date
parts = [india.withColumn("source", F.lit("india")),
         uk.withColumn("source", F.lit("uk")),
         uae.withColumn("source", F.lit("uae"))]
sol = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), parts) \
        .withColumn("sale_date", F.to_date("sale_date"))

# 2. Defaults per source
sol = sol.withColumn("currency", F.coalesce(
    "currency", F.when(F.col("source") == "india", "INR").when(F.col("source") == "uk", "GBP")))
sol.show()

# 3. Totals per source
sol.groupBy("source").agg(F.sum("amount").alias("total")).show()

# 4. Rows in more than one source
sol.groupBy("store", "sale_date", "amount").agg(F.countDistinct("source").alias("n_sources")) \
   .filter("n_sources > 1").show()

# 5. Set difference (same column order needed for positional set operations)
india.subtract(uk.select("store", "sale_date", "amount")).show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>union</code> matches by <b>position</b>, <code>unionByName</code> by <b>name</b>. Prefer <code>unionByName</code></li><li><code>allowMissingColumns=True</code> fills missing columns with nulls</li><li>Union keeps duplicates (<code>UNION ALL</code>). Add <code>.distinct()</code> for SQL <code>UNION</code></li><li>Union is narrow: no shuffle, partitions are concatenated</li><li>Cast types explicitly before unioning</li><li>Add a <code>source</code> column so rows stay traceable</li><li><code>intersect</code>, <code>subtract</code> (distinct), <code>intersectAll</code>, <code>exceptAll</code> (keep duplicates) shuffle</li></ul>
</div>

| Task | Code |
|---|---|
| By name | `a.unionByName(b)` |
| Missing columns | `a.unionByName(b, allowMissingColumns=True)` |
| Many | `reduce(lambda x, y: x.unionByName(y), dfs)` |
| Deduplicated | `a.union(b).distinct()` |
| In both | `a.intersect(b)` |
| In a not b | `a.subtract(b)`, `a.exceptAll(b)` |
| SQL | `UNION ALL`, `UNION`, `INTERSECT`, `EXCEPT` |

## Git commit

```
git add 01_spark_basics/18_union.ipynb
git commit -m "add 18 union notebook"
```